# Parallel functionality of Numba

 - Elwin van 't Wout
 - Pontificia Universidad Católica de Chile
 - IMT3870
 - 29-8-2022

Sum the values of a vector and compare the timing between parallelised versions.

In [ ]:
import numpy as np
from numba import jit, prange

In [ ]:
def sum_vector_python(a):
    s = 0
    for i in range(a.size):
        s += a[i]
    return s   

In [ ]:
def sum_vector_numpy(a):
    s = np.sum(a)
    return s   

For Numba, we can use exactly the same function as before but with the Numba decorator added. As the first version, we use the Numba optimisation (```nopython=True```) but without parallelisation (```parallel=False```).

In [ ]:
@jit(nopython=True, parallel=False)
def sum_vector_numba_serial(a):
    s = 0
    for i in range(a.size):
        s += a[i]
    return s   

Adding the parallel option to the Numba decorator makes Numba search for parts of the code than can be parallelised. Add the option ```parallel=True``` for automatic parallelisation. This will only work when ```nopython=True```.

In [ ]:
@jit(nopython=True, parallel=True)
def sum_vector_numba_parallel(a):
    s = 0
    for i in range(a.size):
        s += a[i]
    return s   

Instead of letting Numba search for parallelisation opportunities, you can explicitly state that a for loop needs to be parallelised. Use the function ```prange()``` instead of the standard ```range()```in the for loop. In this case, Numba automatically detects that the variable ```s``` for the sum is a shared variable.

In [ ]:
@jit(nopython=True, parallel=True)
def sum_vector_numba_prange(a):
    s = 0
    for i in prange(a.size):
        s += a[i]
    return s   

Let us create a vector with elements $0,1,2,\dots,n$ and calculate the sum.

In [ ]:
n = int(1e7)
vec = np.arange(n)

Before performing the timings, call the Numba functions once, so that they are compiled

In [ ]:
print("Sum of vector with serial Numba:", sum_vector_numba_serial(vec))
print("Sum of vector with parallel Numba:", sum_vector_numba_parallel(vec))
print("Sum of vector with prange Numba:", sum_vector_numba_prange(vec))

Numba might give warnings when it is not able to perform the requested optimisation of the code.

In [ ]:
%%timeit
sum_vector_python(vec)

In [ ]:
%%timeit
sum_vector_numpy(vec)

In [ ]:
%%timeit
sum_vector_numba_serial(vec)

In [ ]:
%%timeit
sum_vector_numba_parallel(vec)

In [ ]:
%%timeit
sum_vector_numba_prange(vec)

The number of threads used by Numba is stored in global variables.

In [ ]:
from numba import config
print("The number of available CPUs detected by Numba is:", config.NUMBA_DEFAULT_NUM_THREADS)
print("The number of threads used by Numba is:", config.NUMBA_NUM_THREADS)

The number of threads used by Numba can be changed manually.

In [ ]:
from numba import set_num_threads, get_num_threads
set_num_threads(2)
print("The current number of threads used by Numba is:", get_num_threads())